# How many ways can a sum occur?
### Convolution, ordered packing, and additive energy

Place every ordered pair $(a,b)$ from two finite integer sets on a grid. Points on
one diagonal have the same sum. Count them, keep those counts as an arrangement,
and ask what happens when many pairs meet at the same coordinate.

The same measurements will describe discrete convolution and count equal-sum
quadruples. This lesson connects a familiar computational operation to a geometric
question about the structure of sets.

[Lesson notes](../docs/lessons/07_additive_structure.md) · [Setup](README.md)

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import plotly.io as pio
from IPython.display import Video, display
from kaleion import Collection, Product, F, Motion, Sweep, Workspace, param
from kaleion.viewers.plotly import animation_figure
from kaleion.viewers.video import write_mp4

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
sys.path.insert(0, str(ROOT / "notebooks"))
from lesson_views import COLORS, cell_panels, profiles, replay, save_figures
pio.renderers.default = "plotly_mimetype+notebook"
OUTPUT = ROOT / "build/notebooks/additive-structure"
OUTPUT.mkdir(parents=True, exist_ok=True)

A_VALUES, B_VALUES = [0, 1, 2, 3], [0, 1, 2, 3]
for values in (A_VALUES, B_VALUES):
    assert all(isinstance(v, int) and not isinstance(v, bool) for v in values)
    assert len(values) == len(set(values)), "These are sets; repeated labels would describe multiplicities."
    assert len(values) <= 6 and all(abs(v) <= 20 for v in values), "Use small sets for these full views."
LOW = min(A_VALUES) + min(B_VALUES) if A_VALUES and B_VALUES else 0
HIGH = max(A_VALUES) + max(B_VALUES) if A_VALUES and B_VALUES else 0
assert HIGH - LOW <= 30, "Choose a short displayed sum interval."

## 1 · Construct a pair arrangement and a measurement domain

\[
r_{A,B}(s)=\#\{(a,b)\in A\times B:a+b=s\}.
\]

We declare every integer bin between the smallest and largest possible sum.
Measuring on a **pair × bin** domain retains these bins even when the count is zero.
`Product(left=left, right=right)` names each factor. A second product names
`pair` and `bin`; its retained `F.bin` axis keeps zero groups. The recipe derives
source sizes symbolically and leaves the relation and bin coverage visible.

In [ ]:
def integer_pairs(left_values, right_values):
    left, right = Collection.literal(left_values), Collection.literal(right_values)
    product = Product(left=left, right=right)
    return (product.domain.annotate(a=product.read("left"), b=product.read("right"))
            .annotate(total=F.a + F.b, pair_key=F.key)
            .with_values(F.total).arrange(F.a, F.b))

def representation_counts(pairs, lower, upper):
    bins = Collection.sequence(upper - lower + 1, start=lower)
    product = Product(pair=pairs, bin=bins)
    measurement_domain = product.domain.annotate(
        pair_key=product.read("pair", F.pair_key), s=product.read("bin"),
        total=product.read("pair", F.total))
    matching = measurement_domain.where(F.total == F.s)
    counts = matching.count(by=F.bin).annotate(s=bins.bind(on=F.bin, key=F.index)).arrange(F.s, F.value)
    return counts, matching

pairs = integer_pairs(A_VALUES, B_VALUES)
counts, measurement = representation_counts(pairs, LOW, HIGH)
workspace = Workspace({"pairs": pairs, "counts": counts, "measurement": measurement})
assert not workspace.state.errors, dict(workspace.state.errors)
state = workspace.state
assert sum(state.results["counts"].values) == len(A_VALUES) * len(B_VALUES)
count_plot = profiles([state.results["counts"]], ["Representations r(s)"], keys=["s"],
                       title="Count the ordered pairs giving each sum")
count_plot.show()

## 2 · Sweep an incidence lens across the sums

Scrub the exact integer cases. The selected diagonal is a subset of ordered pair
occurrences: $(a,b)$ and $(b,a)$ are different when both are present.

If $\mathbf1_A$ and $\mathbf1_B$ are indicator functions, then

\[
r_{A,B}(s)=\sum_t\mathbf1_A(t)\mathbf1_B(s-t)
=(\mathbf1_A*\mathbf1_B)(s).
\]

This is discrete convolution. The notebook uses ordinary integer addition;
cyclic convolution would require an explicitly declared modulus and residue bins.

In [ ]:
sum_lens = pairs.where(F.total == param("s"))
sum_cases = Sweep(sum_lens, "s", range(LOW, HIGH + 1))
case_snapshots = [sum_cases.at(i) for i in range(HIGH - LOW + 1)]
captions = [f"sum {s} · {snapshot.cardinality} ordered pairs"
            for s, snapshot in zip(range(LOW, HIGH + 1), case_snapshots)]
assert [snapshot.cardinality for snapshot in case_snapshots] == state.results["counts"].values.tolist()
lens_plot = animation_figure(case_snapshots, labels=captions, title="A diagonal lens measures convolution",
                             duration=600, show_values=True)
lens_plot.show()

## 3 · Count earlier matches to assign an ordered slot

The map $(a,b)\mapsto(a+b,0)$ brings equal sums together. It is not a permutation
of spatial slots: several occurrences can coincide while keeping distinct identities.

To make multiplicity visible again, assign each pair a rank among earlier pairs
with the same sum. The order is the declared `pair_key`, retained independently
of later placement. Another incidence count supplies that rank:

\[
\rho(q)=\#\{r:r<q,\ s(r)=s(q)\}.
\]

Use $\rho(q)$ as the new vertical position. Every sum becomes a stack of height
$r_{A,B}(s)$, including an explicit empty bin when no pair has that sum.

`group_by(F.total)` chooses the fibers; `order_by(F.pair_key)` chooses their
member order; `ranks(key=F.pair_key)` measures zero-based predecessor counts.
The named height binding then drives `arrange(x=..., y=...)`. Rank evaluation
sorts within groups and stores compact prefix evidence. It does not expand the
predecessor relation into a pair matrix. The separate pair-of-pairs incidence
below is retained for the independent additive-energy count.

In [ ]:
groups = pairs.group_by(F.total)
ranks = groups.order_by(F.pair_key).ranks(key=F.pair_key)
height = ranks.bind(on=F.pair_key, key=F.key)
collapsed = pairs.arrange(x=F.total, y=0)
stacked = collapsed.arrange(x=F.total, y=height)

motion_workspace = Workspace({"pairs": pairs})
collapse = motion_workspace.set("pairs", collapsed, motion=Motion())
stack = motion_workspace.set("pairs", stacked, motion=Motion.arc(height=1.2))
stacked_state = motion_workspace.state
assert not stacked_state.errors
assert stacked_state.results["pairs"].ids == state.results["pairs"].ids
expected_slots = {(int(s), rank) for s, count in zip(state.results["counts"].fields["s"], state.results["counts"].values)
                  for rank in range(int(count))}
assert {tuple(map(int, point)) for point in stacked_state.results["pairs"].positions} == expected_slots
motion_workspace.capture("Equal-sum occurrences remain distinct; predecessor counts place them in consecutive slots.")
unstack, uncollapse = motion_workspace.undo(), motion_workspace.undo()
samples, labels = [], []
for name, transition in (("bring equal sums together", collapse), ("stack by measured rank", stack),
                         ("undo stacking", unstack), ("restore the pair arrangement", uncollapse)):
    for t in np.linspace(0, 1, 21):
        samples.append(transition.frame("pairs", float(t)))
        labels.append(f"{name} · {t:.0%}")
np.testing.assert_array_equal(samples[0].positions, samples[-1].positions)
for outward, inward in ((collapse, uncollapse), (stack, unstack)):
    np.testing.assert_array_equal(outward.frame("pairs", .25).positions, inward.frame("pairs", .75).positions)
colors = {oid: COLORS[(int(value) - LOW) % len(COLORS)]
          for oid, value in zip(state.results["pairs"].ids, state.results["pairs"].values)}
motion_plot = replay(samples, labels, title="Coincidence does not erase occurrences · counts make stacks", colors_by_id=colors)
motion_plot.show()

## 4 · Count the collisions between representations

For a fixed sum $s$, there are $r(s)$ choices for one representing pair and $r(s)$
for a second. That gives $r(s)^2$ equal-sum quadruples. Therefore

\[
\boxed{\sum_s r_{A,B}(s)^2
=\#\{(a,b,c,d)\in A\times B\times A\times B:a+b=c+d\}.}
\]

When $A=B$, this is the **additive energy** $E(A)$. Build squares whose side lengths
are the measured $r(s)$, and count their cells. Compare that count with the incidence
on the pair-of-pairs domain. The equality follows by grouping the same quadruples
by their common sum; it is not inferred from the rendered marker areas.

In [ ]:
pair_product = Product(left=pairs, right=pairs)
pair_pairs = pair_product.domain.annotate(
    left_total=pair_product.read("left", F.total),
    right_total=pair_product.read("right", F.total))
equal_sums = pair_pairs.where(F.left_total == F.right_total)

MAX_REP = min(len(A_VALUES), len(B_VALUES))  # Unique input labels ensure this finite bound.
side = counts.bind(on=F.i, key=F.bin)
square_domain = (Collection.grid(HIGH - LOW + 1, MAX_REP, MAX_REP, values=1)
                 .arrange(F.i * (MAX_REP + 2) + F.j, F.k))
energy_squares = square_domain.where((F.j < side) & (F.k < side))
energy = counts.sum(value=F.value ** 2)
collision_workspace = Workspace({"counts": counts, "ranks": ranks, "squares": energy_squares,
                                 "energy": energy, "quadruples": equal_sums.count(),
                                 "square_count": energy_squares.count(), "measurement": measurement,
                                 "pairs": pairs})
assert not collision_workspace.state.errors, dict(collision_workspace.state.errors)
cr = collision_workspace.state.results
E = int(cr["energy"].values[0])
assert E == int(cr["quadruples"].values[0]) == int(cr["square_count"].values[0])
energy_plot = cell_panels([cr["squares"]], ["One measured square for each sum"],
                          title=f"Count equal-sum quadruples · total {E}", height=410)
energy_plot.data[0].marker.size = 14
for bin_index, (s, count) in enumerate(zip(cr["counts"].fields["s"], cr["counts"].values)):
    energy_plot.add_annotation(x=bin_index * (MAX_REP + 2) + max(int(count) - 1, 0) / 2,
                              y=MAX_REP + .6, text=f"{int(s)}: {int(count)}²", showarrow=False)
energy_plot.update_xaxes(range=[-1, (HIGH - LOW + 1) * (MAX_REP + 2)], showticklabels=False,
                         title_text="sum s: measured square area r(s)²")
energy_plot.update_yaxes(range=[-1, MAX_REP + 2], showticklabels=False, title_text="")
energy_plot.show()
print("Sum of squared representation counts = direct quadruple count =", E)

## 5 · Compare structure with scattering

Both sets below have four elements and sixteen ordered pairs. For $\{0,1,2,3\}$,
many pairs share sums; its energy is 44. For $\{0,1,3,7\}$ the energy is 28.
Its empty integer bins remain visible. More concentrated representation counts
give a larger sum of squares when the total number of representations is fixed.

This is a finite comparison, not a claim that a single energy value uniquely
identifies a set or determines all its additive structure. For a broader treatment,
see [Yufei Zhao, Structure of Set Addition](https://yufeizhao.com/gtacbook/7.pdf).

In [ ]:
structured_pairs = integer_pairs([0, 1, 2, 3], [0, 1, 2, 3])
scattered_pairs = integer_pairs([0, 1, 3, 7], [0, 1, 3, 7])
structured_counts, _ = representation_counts(structured_pairs, 0, 14)
scattered_counts, _ = representation_counts(scattered_pairs, 0, 14)
comparison_workspace = Workspace({"structured": structured_counts, "scattered": scattered_counts,
                                  "E_structured": structured_counts.sum(value=F.value ** 2),
                                  "E_scattered": scattered_counts.sum(value=F.value ** 2)})
assert not comparison_workspace.state.errors
rr = comparison_workspace.state.results
assert int(rr["E_structured"].values[0]) == 44
assert int(rr["E_scattered"].values[0]) == 28
assert sum(rr["structured"].values) == sum(rr["scattered"].values) == 16
comparison_plot = profiles([rr["structured"], rr["scattered"]],
    ["{0,1,2,3} · energy 44", "{0,1,3,7} · energy 28"], keys=["s", "s"],
    title="Same number of pairs · different distributions of coincidences")
comparison_plot.update_yaxes(range=[0, 4.8], dtick=1)
comparison_plot.update_xaxes(dtick=2)
comparison_plot.show()

## 6 · Inspect a sum, save the motion, and try another set

The count contributors are occurrences in the pair × bin measurement domain. Each
records a `pair_key` that identifies the original ordered pair. The saved explanation
keeps both levels. Change the input sets, including gaps or negative integers, then
rerun the notebook. Duplicated input labels are excluded here so set cardinality
and occurrence multiplicity cannot be silently confused.

The MP4 below is the existing 2D raster adapter sampling these same recorded frames;
it is not a screen recording of Plotly. Interactive HTML exports retain the controls.

In [ ]:
SUM = min(HIGH, max(LOW, 3))
measured_source = state.results["measurement"].source
measurement_index = {oid: index for index, oid in enumerate(measured_source.ids)}
source_pairs = state.results["pairs"]
pair_index = {int(key): index for index, key in enumerate(source_pairs.fields["pair_key"])}
contributors = state.results["counts"].contributor_ids(SUM - LOW)
explanation = {"sum": SUM, "count": len(contributors), "contributors": []}
for oid in contributors:
    source_index = pair_index[int(measured_source.fields["pair_key"][measurement_index[oid]])]
    explanation["contributors"].append({"measurement_occurrence": oid,
        "pair_occurrence": source_pairs.ids[source_index],
        "pair": [int(source_pairs.fields[name][source_index]) for name in ("a", "b")]})
assert explanation["count"] == int(state.results["counts"].values[SUM - LOW])
print("Representations of", SUM, ":", [item["pair"] for item in explanation["contributors"]])

movie = write_mp4(samples, OUTPUT / "sum-stacks.mp4", labels=labels,
                   title="Equal sums: collapse, count ranks, stack, undo", fps=24)
display(Video(str(movie), embed=True))
collision_workspace.capture("Convolution counts, measured ranks, and equal-sum quadruples are explicit constructions.")
save_figures(OUTPUT, {"representation-counts": count_plot, "sum-lens": lens_plot,
    "sum-stacks": motion_plot, "energy-squares": energy_plot, "structure-comparison": comparison_plot})
for name, investigation in (("collisions", collision_workspace), ("motion", motion_workspace),
                            ("comparison", comparison_workspace)):
    payload = investigation.to_json()
    (OUTPUT / f"{name}-workspace.json").write_text(payload)
    assert not Workspace.from_json(payload).state.errors
restored = Workspace.from_json(collision_workspace.to_json())
assert restored.state.results["counts"].contributor_ids(SUM - LOW) == contributors
(OUTPUT / "sum-explanation.json").write_text(json.dumps(explanation, indent=2))
(OUTPUT / "checks.json").write_text(json.dumps({"A": A_VALUES, "B": B_VALUES,
    "sums": list(map(int, cr["counts"].fields["s"])), "counts": list(map(int, cr["counts"].values)),
    "energy": E, "comparison_energies": [44, 28], "motion_frames": len(samples)}, indent=2))
print("Saved five offline figures, the 2D MP4, workspaces, and the sum explanation to", OUTPUT)

The recurring operation is an **ordered rank inside a group**. Its declaration
now names grouping, member order, and the output key explicitly. The evaluator
sorts members and stores a compact predecessor range for each result. The earlier
Young-diagram prefix-sum recipe remains a separate opportunity for simplification.

This does not make every pair-domain construction unnecessary: additive energy
still asks a question about pairs of representations. The distinction is between
the mathematical domain we want to inspect and an avoidable expansion used to
compute ranks. See the [authoring guide](../docs/AUTHORING.md) and
[review notes](../docs/lessons/REVIEW_NOTES.md).

Next: [counting lattice points as shapes grow](08_ehrhart_counts.ipynb).